In [1]:
 import numpy as np
 import pandas as pd
 import matplotlib.pyplot as plt
 import seaborn as sns

In [2]:
from sklearn.preprocessing import MultiLabelBinarizer  
from sklearn.cluster import KMeans                      
from sklearn.metrics import silhouette_score            
from sklearn.decomposition import PCA                   

In [3]:
 jobs = pd.read_csv("adzuna_ai_jobs_dach.csv")
 jobs.head()

,id,title,company,location,country,category,contract_type,contract_time,salary_min,salary_max,date_posted,description,role_category,search_query,url
0,de_5708353171,Senior Engineer Machine Learning / AI (w/m/d),HUK-COBURG Versicherungsgruppe,"Coburg, Coburg (Kreis)",Germany,IT-Stellen,NaN,NaN,NaN,NaN,2026-04-23,"Hallo, lass uns WIR sagen! Wir - die HUK-COBUR...",AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/land/ad/5708353171?se=fG...
1,de_5722278756,AI & Machine Learning Engineer (all genders),SThree Germany,"Hausen, Frankfurt am Main",Germany,IT-Stellen,permanent,full_time,80000.0,95000.0,2026-05-08,AI & Machine Learning Engineer (all genders) U...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5722278756?utm_m...
2,de_5680334179,Digital Process Engineer – AI & Automation (KI...,GPE MedTech Systeme GmbH,"Dassow, Nordwestmecklenburg (Kreis)",Germany,Beraterstellen,NaN,NaN,NaN,NaN,2026-03-27,Digital Process Engineer – AI & Automation (KI...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5680334179?utm_m...
3,de_5418313407,Senior AI / Machine Learning Engineer – Enterp...,Nortal,Deutschland,Germany,IT-Stellen,permanent,full_time,NaN,NaN,2025-09-27,Wir bei Nortal unterstützen den öffentlichen S...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5418313407?utm_m...
4,de_5698582069,AI & Machine Learning Engineer – Sensor Fusion...,Collective.work,"Altstadt-Lehel, München",Germany,Sonstige/Allgemeine Stellen,permanent,full_time,65000.0,105000.0,2026-04-14,Job description Salary: 65-105k€ (Annuel) AI &...,AI_Engineering,AI Engineer Machine Learning,https://www.adzuna.de/details/5698582069?utm_m...


In [4]:
 skills = pd.read_csv("adzuna_ai_skills_dach.csv")
 skills.head()

,job_id,skill,country,role_category
0,de_5722278756,git,Germany,AI_Engineering
1,de_5722278756,machine learning,Germany,AI_Engineering
2,de_5722278756,rag,Germany,AI_Engineering
3,de_5680334179,git,Germany,AI_Engineering
4,de_5680334179,machine learning,Germany,AI_Engineering


In [5]:
print("Jobs shape:", jobs.shape)
print("Skills shape:", skills.shape)
print("\nJob columns:", jobs.columns.tolist())

Jobs shape: (485, 15)
Skills shape: (378, 4)

Job columns: ['id', 'title', 'company', 'location', 'country', 'category', 'contract_type', 'contract_time', 'salary_min', 'salary_max', 'date_posted', 'description', 'role_category', 'search_query', 'url']


In [6]:
print("\nMissing values (jobs):")
print(jobs.isnull().sum())


Missing values (jobs):
id                 0
title              0
company            1
location           0
country            0
category           0
contract_type    418
contract_time    269
salary_min       457
salary_max       457
date_posted        0
description        0
role_category      0
search_query       0
url                0
dtype: int64


In [7]:
jobs_with_skills = skills["job_id"].nunique()
print(f"\nJobs with ≥1 skill: {jobs_with_skills} / {len(jobs)}")
print(f"Jobs with 0 skills: {len(jobs) - jobs_with_skills}")


Jobs with ≥1 skill: 252 / 485
Jobs with 0 skills: 233


In [8]:
print("\nSkills per job stats:")
print(skills.groupby("job_id")["skill"].count().describe())


Skills per job stats:
count    252.000000
mean       1.500000
std        0.894873
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max        5.000000
Name: skill, dtype: float64


In [9]:
print("\nTop 20 skills:")
print(skills["skill"].value_counts().head(20))


Top 20 skills:
skill
git                       87
rag                       54
machine learning          32
sap                       26
scala                     25
generative ai             22
agile                     17
api                       16
rpa                       16
llm                       14
computer vision           14
aws                       11
deep learning              8
spark                      5
azure                      5
python                     4
kubernetes                 3
reinforcement learning     2
databricks                 2
mlops                      2
Name: count, dtype: int64


The initial dataset collected via fetch_ai_jobs_and_skills.py contains 485 unique job postings across the DACH region. The Adzuna API caps description length at approximately 500 characters, which limited skill extraction to only 252 of 485 jobs (52%) with a mean of 1.5 skills per job — too sparse for meaningful clustering. To address this, fetch_ai_jobs_and_skills_enriched.py follows each posting's redirect URL, fetches the full job page and re-runs skill extraction on the complete text, saving the results to adzuna_ai_jobs_dach_enriched.csv and adzuna_ai_skills_dach_enriched.csv.

In [10]:
jobs_enriched = pd.read_csv("adzuna_ai_jobs_dach_enriched.csv")
skills_enriched = pd.read_csv("adzuna_ai_skills_dach_enriched.csv")

In [11]:
print("Jobs shape:", jobs_enriched.shape)
print("Skills shape:", skills_enriched.shape)

Jobs shape: (485, 17)
Skills shape: (1470, 4)


In [12]:
print("Description length distribution:")
print(jobs_enriched["description_length"].describe())
print(f"\nFetch failed (<=500 chars): {(jobs_enriched['description_length'] <= 500).sum()}")
print(f"Fetch succeeded (>500 chars): {(jobs_enriched['description_length'] > 500).sum()}")

Description length distribution:
count     485.000000
mean     3402.540206
std      2034.167858
min       500.000000
25%       500.000000
50%      4974.000000
75%      5000.000000
max      5000.000000
Name: description_length, dtype: float64

Fetch failed (<=500 chars): 155
Fetch succeeded (>500 chars): 330


In [13]:
jobs_with_skills = skills_enriched["job_id"].nunique()
print(f"\nJobs with ≥1 skill: {jobs_with_skills} / {len(jobs_enriched)}")
print(f"Jobs with 0 skills: {len(jobs_enriched) - jobs_with_skills}")


Jobs with ≥1 skill: 304 / 485
Jobs with 0 skills: 181


In [14]:
print("\nTop 25 skills:")
print(skills_enriched["skill"].value_counts().head(25))


Top 25 skills:
skill
python              123
machine learning    114
scala               101
generative ai        84
agile                69
aws                  65
llm                  62
ci/cd                58
azure                56
kubernetes           50
docker               49
mlops                43
pytorch              43
gcp                  41
rpa                  38
api                  37
sql                  36
computer vision      33
sap                  32
deep learning        28
tensorflow           26
nlp                  25
statistics           25
git                  25
spark                23
Name: count, dtype: int64


In [6]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
